# 03 — Первый воспроизводимый baseline

Этот notebook строит **честную точку отсчёта** для обычной табличной классификации или регрессии. Он сравнивает no-skill модель и простую модель на одних folds, при этом imputation, scaling и encoding обучаются только внутри train-части каждого fold.

Notebook остаётся оркестратором: конфигурация хранится отдельно, а переиспользуемая логика импортируется из `src/ml_project/baseline.py`.

| Что настраивается | Единый источник истины |
|---|---|
| Файлы, target, key, группы признаков | `src/ml_project/config.py` |
| Контракт основной метрики | `docs/00_problem.md` → `(primary_metric:: ...)` |
| CV, preprocessing, модель и действия записи | `src/ml_project/baseline_config.py` |
| Обоснование validation-протокола | `docs/03_validation.md` |
| Решения о составе признаков | `docs/04_features.md` |

<details>
<summary><strong>Критические предостережения — прочитать перед запуском</strong></summary>

1. **Не заполняйте пропуски до CV.** Медиана, среднее, scaling, encoding и feature selection должны обучаться внутри `Pipeline` на train-fold.
2. **Inference/test не является validation.** Этот notebook не оценивает качество по inference-датасету и не использует его статистики.
3. **Схема split важнее модели.** Для времени, пользователей, устройств, пациентов и повторных объектов обычный random split может давать ложный результат.
4. **Identifier не становится признаком автоматически.** KEY, target и группы `identifier`/`ignored` исключаются из model matrix.
5. **Baseline — не попытка получить лучший score.** Это простая, воспроизводимая нижняя точка сравнения.
6. **Не включайте несколько изменений одновременно.** После baseline каждую существенную идею проверяйте отдельным экспериментом.
7. **Loss-метрики sklearn имеют префикс `neg_`.** В отчёте знак возвращается к обычному виду, а направление показывается как `minimize`.
8. **Запись выключена по умолчанию.** Модель, CSV и Markdown изменяются только явными флагами в `baseline_config.py`.

</details>

> [!note]
> Для текста, изображений, временных рядов и deep learning этот notebook служит архитектурным каркасом. Добавляется новый transformer/estimator, но границы `config → pipeline → CV → artifacts → docs` сохраняются.


## 0. Перед первым запуском

- [ ] В `docs/00_problem.md` заполнено `(primary_metric:: ...)`.
- [ ] В `src/ml_project/config.py` заданы `TARGET`, `KEY` и все `FEATURE_GROUPS`.
- [ ] В `docs/03_validation.md` обоснована схема split.
- [ ] В `src/ml_project/baseline_config.py` заполнен обязательный `TASK_TYPE`.
- [ ] Установлены зависимости: из корня проекта `python -m pip install -r requirements.txt`, либо из notebook `%pip install -r ../requirements.txt`.

Сначала исправьте конфиги, затем запускайте notebook сверху вниз.


In [ ]:
from pathlib import Path
import importlib
import sys

from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").exists() and (candidate / "src").exists()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project import DataCatalog
import ml_project.config as project_config
import ml_project.baseline_config as baseline_config
import ml_project.baseline as baseline_tools

print(f"Корень проекта: {PROJECT_ROOT}")


## 1. Перечитать и проверить конфигурацию

После любого изменения `config.py` или `baseline_config.py` повторно запустите эту ячейку и все ячейки ниже. **Restart kernel не требуется.**


In [ ]:
project_config = importlib.reload(project_config)
baseline_config = importlib.reload(baseline_config)
baseline_tools = importlib.reload(baseline_tools)

DATASETS = project_config.DATASETS
FEATURE_GROUPS = project_config.FEATURE_GROUPS
INFERENCE_DATASET = project_config.INFERENCE_DATASET
KEY = project_config.KEY
RAW_DIR = project_config.RAW_DIR
TARGET = project_config.TARGET
TRAIN_DATASET = project_config.TRAIN_DATASET

settings = baseline_tools.settings_from_module(baseline_config)
display(baseline_tools.settings_report(settings))

config_warnings = baseline_tools.validate_baseline_settings(settings)
for warning in config_warnings:
    print(f"ПРЕДУПРЕЖДЕНИЕ: {warning}")
print("Конфигурация baseline прошла проверку.")


## 2. Загрузить train и зафиксировать версию

Загружается исходный train из `data/raw`. Notebook не создаёт заранее «очищенный» общий CSV: model-ready преобразования должны оставаться внутри pipeline.


In [ ]:
catalog = DataCatalog(PROJECT_ROOT, RAW_DIR, DATASETS)
catalog.validate()

available_datasets = set(catalog.available_names())
if TRAIN_DATASET not in available_datasets:
    raise KeyError(f"Train dataset {TRAIN_DATASET!r} is unavailable.")

train_frame = catalog.load(TRAIN_DATASET).copy()
inference_frame = (
    catalog.load(INFERENCE_DATASET).copy()
    if INFERENCE_DATASET and INFERENCE_DATASET in available_datasets
    else None
)

file_report = catalog.file_report()
train_file = file_report.loc[file_report["dataset"] == TRAIN_DATASET].iloc[0]
dataset_version = str(train_file["sha256"])

print(f"Train: {train_frame.shape[0]:,} строк × {train_frame.shape[1]:,} столбцов")
print(f"Версия train (SHA-256): {dataset_version}")
if inference_frame is not None:
    print(f"Inference: {inference_frame.shape[0]:,} строк × {inference_frame.shape[1]:,} столбцов")
else:
    print("Inference-датасет не настроен или отсутствует — для CV это не проблема.")


## 3. Построить явный feature plan

Типы признаков берутся только из `FEATURE_GROUPS`, а не угадываются по dtype. Таблица ниже показывает каждый столбец и причину его использования или исключения.


In [ ]:
feature_plan = baseline_tools.resolve_feature_plan(
    train_frame,
    FEATURE_GROUPS,
    target=TARGET,
    key=KEY,
    settings=settings,
)
display(feature_plan.to_frame())

for warning in feature_plan.warnings:
    print(f"ПРЕДУПРЕЖДЕНИЕ: {warning}")

if inference_frame is not None:
    inference_warnings = baseline_tools.validate_inference_schema(
        inference_frame,
        feature_plan,
        strict=settings.require_inference_features,
    )
    for warning in inference_warnings:
        print(f"ПРЕДУПРЕЖДЕНИЕ: {warning}")

prepared = baseline_tools.prepare_training_data(
    train_frame,
    target=TARGET,
    plan=feature_plan,
    settings=settings,
)
print("Model matrix:", prepared.X.shape)
print("Числовые признаки:", len(feature_plan.numeric))
print("Категориальные признаки:", len(feature_plan.categorical))
print("Исключённые признаки:", len(feature_plan.excluded))


## 4. Собрать preprocessing

`SimpleImputer`, scaler и `OneHotEncoder` пока **не обучаются**. Они становятся частью общего sklearn `Pipeline` и будут fit-иться заново внутри каждого train-fold.

`remainder="drop"` защищает от случайного попадания незаявленного столбца в модель.


In [ ]:
preprocessor = baseline_tools.build_tabular_preprocessor(
    settings,
    feature_plan,
)
display(baseline_tools.preprocessing_report(settings, feature_plan))
print(preprocessor)


## 5. Собрать metric и CV

Основная метрика читается из `docs/00_problem.md`. `PRIMARY_SCORER` в baseline-конфиге является только техническим адаптером на случай, если человеческое название не совпадает с именем scorer в sklearn.


In [ ]:
scoring_plan = baseline_tools.resolve_scoring_plan(
    PROJECT_ROOT,
    settings,
)
cv, resolved_cv_strategy = baseline_tools.build_cv_splitter(
    settings,
    prepared.y,
)
cv_description = baseline_tools.cv_protocol_description(
    settings,
    resolved_cv_strategy,
)

display(scoring_plan.to_frame())
print("Validation:", cv_description)
if settings.cv_strategy == "auto":
    print("ПРЕДУПРЕЖДЕНИЕ: auto-стратегию нужно заменить явной после фиксации validation-протокола.")


## 6. Собрать сравнимые baseline-модели

Обе модели получают один и тот же preprocessing и один и тот же CV:

- `dummy` — no-skill нижняя граница;
- простая модель — логистическая регрессия для классификации или Ridge для регрессии; в отчёте используется её настоящее имя.

Более мощная sklearn-совместимая модель позднее может использовать тот же `preprocessor`, `build_model_pipeline` и `evaluate_models_cv`.


In [ ]:
models = {}
if settings.run_dummy_baseline:
    models["dummy"] = baseline_tools.build_model_pipeline(
        preprocessor,
        baseline_tools.build_dummy_estimator(settings),
    )

simple_model_name = baseline_tools.resolved_model_name(settings)
models[simple_model_name] = baseline_tools.build_model_pipeline(
    preprocessor,
    baseline_tools.build_simple_estimator(settings),
)

print("Модели:", ", ".join(models))
print("Simple estimator:", simple_model_name)


## 7. Выполнить cross-validation

Это единственная ячейка, где рассчитывается baseline-score. Если она падает, не заменяйте ошибку на `NaN`: исправьте feature schema, split, scorer или preprocessing.


In [ ]:
evaluation = baseline_tools.evaluate_models_cv(
    models,
    prepared,
    cv=cv,
    scoring=scoring_plan,
    settings=settings,
)

validation_folds = evaluation.fold_scores[
    evaluation.fold_scores["split"] == "validation"
]
fold_table = validation_folds.pivot_table(
    index=["model", "fold"],
    columns="metric",
    values="value",
).reset_index()

display(fold_table.round(4))
display(evaluation.summary.round(4))


## 8. Интерпретировать результат до сохранения

Проверьте не только среднее:

1. Простая модель должна осмысленно сравниваться с `dummy`.
2. Большой `std` между folds означает нестабильность оценки или неоднородные данные.
3. Подозрительно высокий результат требует повторной проверки leakage.
4. Слабый результат не означает, что notebook сломан: baseline фиксирует реальную стартовую точку.
5. Нельзя утверждать, что отдельный preprocessing-шаг «помог», пока он не проверен контролируемым экспериментом.


In [ ]:
primary_result = evaluation.primary_summary().copy()
primary_result["mean ± std"] = primary_result.apply(
    lambda row: f"{row['mean']:.4f} ± {row['std']:.4f}",
    axis=1,
)
display(
    primary_result[
        ["model", "metric", "direction", "mean ± std", "min", "max", "fit_seconds_mean"]
    ].round(4)
)


## 9. Просмотреть график каждой метрики

Для каждой основной и дополнительной метрики строится отдельный график значений на validation-folds. Линии показывают разброс между folds, а пунктир — среднее модели. Эти же Figure будут сохранены как PNG, если включены `SAVE_ARTIFACTS` и `SAVE_METRIC_FIGURES`.


In [ ]:
metric_figures = baseline_tools.build_metric_figures(evaluation, scoring_plan)
for metric_key, figure in metric_figures.items():
    print(f"Метрика: {scoring_plan.labels[metric_key]}")
    display(figure)


## 10. Явно сохранить артефакты и обновить отчёты

По умолчанию эта ячейка ничего не записывает: основной переключатель — `SAVE_ARTIFACTS`.

В `src/ml_project/baseline_config.py` доступны действия:

- `SAVE_ARTIFACTS = True` — сохранить CV-таблицы, metadata и PNG каждой метрики;
- `SAVE_METRIC_FIGURES = True` — сохранить по одному графику на каждую configured metric;
- `SAVE_FINAL_MODEL = True` — дополнительно fit на всём train и сохранить `model.joblib`;
- `SYNC_EXPERIMENT_NOTE = True` — создать/обновить карточку `EXPERIMENT_NOTE`, встроить графики и Markdown-таблицы;
- `SYNC_DOCS = True` — обновить auto-блоки в Validation и Experiments;
- `ALLOW_OVERWRITE = True` — безопасно повторить тот же run: заменяются только известные сгенерированные файлы, ручные файлы сохраняются.

> [!warning] Новый смысл эксперимента — новый `RUN_NAME` и новая карточка. `ALLOW_OVERWRITE=True` предназначен для повторного запуска той же конфигурации, а не для стирания истории разных экспериментов.

После изменения конфига повторно запустите ячейку **1** и все ячейки ниже неё.


In [ ]:
saved_run = None
write_requested = settings.save_artifacts or settings.save_final_model

if write_requested:
    saved_run = baseline_tools.save_baseline_run(
        PROJECT_ROOT, settings, evaluation, feature_plan, scoring_plan,
        dataset_version=dataset_version, cv_description=cv_description,
        final_pipeline=models[simple_model_name], data=prepared,
        metric_figures=metric_figures,
    )
    print("Артефакты:", saved_run.run_dir.relative_to(PROJECT_ROOT))
    print("CV folds:", saved_run.fold_scores_path.name)
    print("CV summary:", saved_run.summary_path.name)
    print("Metadata:", saved_run.metadata_path.name)
    for metric_key, figure_path in (saved_run.metric_figure_paths or {}).items():
        print(f"График {scoring_plan.labels[metric_key]}:", figure_path.name)
    if saved_run.model_path is not None:
        print("Final model:", saved_run.model_path.name)
else:
    print("SAVE_ARTIFACTS=False и SAVE_FINAL_MODEL=False — файлы не записаны.")

if settings.sync_experiment_note:
    if saved_run is None:
        print("SYNC_EXPERIMENT_NOTE пропущен: сначала включите SAVE_ARTIFACTS.")
    else:
        updated_experiment = baseline_tools.sync_baseline_experiment_note(
            PROJECT_ROOT, settings, evaluation, scoring_plan, saved_run,
            dataset_version=dataset_version[:12] + "…",
            cv_description=cv_description, model_name=simple_model_name,
        )
        print("Обновлена карточка:", settings.experiment_note, updated_experiment)

if settings.sync_docs:
    experiment_note = (
        settings.experiment_note
        if (PROJECT_ROOT / settings.experiment_note).exists() else None
    )
    updated = baseline_tools.sync_baseline_docs(
        PROJECT_ROOT, evaluation, scoring_plan,
        dataset_version=dataset_version[:12] + "…",
        cv_description=cv_description, model_name=simple_model_name,
        experiment_note=experiment_note,
    )
    print("Обновлены auto-блоки:", updated)
else:
    print("SYNC_DOCS=False — stage docs не изменены.")


## 11. После baseline

1. Откройте автоматически созданную карточку из `EXPERIMENT_NOTE`: там уже встроены все графики, сводки и значения по folds.
2. Проверьте сводки в `docs/03_validation.md` и `docs/05_experiments.md`.
3. Запишите активный preprocessing и исключения в `docs/04_features.md`.
4. Только затем создавайте одну проверяемую гипотезу и меняйте один основной фактор.
5. Inference/submission выполняйте отдельным явным шагом после выбора модели; этот notebook намеренно не использует test для оценки.

### Как масштабировать

- **Сильнее tabular model:** передайте любой sklearn-совместимый estimator в `build_model_pipeline`.
- **Новый preprocessing:** добавьте transformer в reusable-модуль, а не функцию внутри notebook.
- **Text/image/deep learning:** сохраните контракт split, metric, artifact metadata и сравнение с этим baseline, заменив только data/model adapter.
- **Tracker/MLflow:** логируйте те же `RUN_NAME`, dataset hash, CV protocol, config и metrics — notebook уже формирует эти сущности отдельно.
